### 1. 📥​ Imports

In [22]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [23]:
import sys, os
from pathlib import Path
import pandas as pd

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

from tcibench.preparation import (Dataset, DatasetMetadata, load_dataset, load_metadata)

from tcibench.generation import (
    generate_with_synthcity,
    balance_data
)

### 2. ⚙️​ Configs

In [24]:
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "tcibench" / "preparation" / "metadata.json").is_file()),
    None,
)

In [25]:
# Reproducibility
SPLIT_SEED = 42
SAMPLING_SEED = 42
GENERATION_SEED = 42

# Generation protocol
EXAMPLES_PER_CLASS = 10
RECORDS_PER_BATCH = 10

📂​ Datasets

In [26]:
DATASETS = {
    "diabetes": "Diabetes",
    "sick": "Thyroid Sick",
    "predictive_maintenance": "Predictive Maintenance",
    "adult": "Adult Income"
}

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / f"split_seed_{SPLIT_SEED}"

RAW_PATHS = {
    "diabetes": RAW_ROOT / "diabetes.csv",
    "sick": RAW_ROOT / "sick.arff",
    "predictive_maintenance": RAW_ROOT / "predictive_maintenance.csv",
    "adult": RAW_ROOT / "adult.csv"
}

🤖 Models

In [27]:
GENERATIVE_MODELS = ["ddpm", "ctgan", "tvae"]
CLASSICAL_METHODS = ["ros", "rus", "smotenc"]

### 3. 📂​ Dataset

In [28]:
def inspect_dataset(
    data: pd.DataFrame,
    metadata: DatasetMetadata,
    cleaned: pd.DataFrame,
    cleaning_stats: dict,
):
    """Inspect a dataset before running the experiment."""

    print(f"Dataset: {metadata.display_name}")
    print(f"Target: {metadata.target_column}")

    print("\nRaw data")
    print(f"Shape: {data.shape}")
    display(data.head())

    print("\nCleaning")
    print(f"Shape after cleaning: {cleaned.shape}")
    print(f"Removed columns: {cleaning_stats['dropped_columns']}")
    print(f"Removed rows with missing values: {cleaning_stats['removed_missing_rows']}")
    print(f"Removed duplicate rows: {cleaning_stats['removed_duplicate_rows']}")

    print("\nCleaned data")
    display(cleaned.head())

    print("\nTarget distribution")
    target = metadata.target_column
    display(pd.DataFrame({
        "count": cleaned[target].value_counts(),
        "proportion": cleaned[target].value_counts(normalize=True).mul(100).round(2),
    }))

    target_counts = cleaned[target].value_counts()
    print(f"Imbalance ratio: {target_counts.max() / target_counts.min():.2f}")

In [29]:
DATASET_ID = "adult"

metadata = load_metadata(DATASET_ID)
data = load_dataset(RAW_PATHS[DATASET_ID])

dataset = Dataset(data, metadata)

cleaned, cleaning_report = dataset.preprocessing.clean(data)
folds = dataset.preprocessing.split(cleaned)

inspect_dataset(data, metadata, cleaned, cleaning_report)

train, test, audit = folds[0]

counts = train[metadata.target_column].value_counts()
majority_class = counts.idxmax()
minority_class = counts.idxmin()

majority_count = int(counts[majority_class])
minority_count = int(counts[minority_class])
records_to_generate = majority_count - minority_count
imbalance_ratio = majority_count / minority_count

print(f"Dataset: {metadata.display_name}")
print(f"Cleaned: {cleaned.shape}")
print(f"Fold 1 — Train: {train.shape} | Test: {test.shape}")
print(f"Majority: {majority_class} ({majority_count})")
print(f"Minority: {minority_class} ({minority_count})")
print(f"Imbalance ratio: {imbalance_ratio:.2f}")
print(f"Records to generate: {records_to_generate}")

Dataset: Adult Income
Target: income

Raw data
Shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K



Cleaning
Shape after cleaning: (45175, 15)
Removed columns: []
Removed rows with missing values: 3620
Removed duplicate rows: 47

Cleaned data


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K



Target distribution


,count,proportion
income,,
<=50K,33973,75.2
>50K,11202,24.8


Imbalance ratio: 3.03
Dataset: Adult Income
Cleaned: (45175, 15)
Fold 1 — Train: (36140, 15) | Test: (9035, 15)
Majority: <=50K (27178)
Minority: >50K (8962)
Imbalance ratio: 3.03
Records to generate: 18216


### 4. 🤖 Models generation

In [30]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

import torch
from synthcity.utils.constants import DEVICE

print("PyTorch:", torch.cuda.is_available())
print("Synthcity:", DEVICE)

True
NVIDIA GeForce RTX 4060 Ti
PyTorch: True
Synthcity: cuda


In [31]:
OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "generation"
report_path = (
    OUTPUT_ROOT / metadata.dataset_id / "baselines" / "generative_stats.json"
)
report_path.parent.mkdir(parents=True, exist_ok=True)

all_stats = []

for fold_id, (train, test, audit) in enumerate(folds):
    counts = train[metadata.target_column].value_counts()
    minority_class = counts.idxmin()
    majority_class = counts.idxmax()

    n_samples = int(
        counts[majority_class] - counts[minority_class]
    )

    for model_name in GENERATIVE_MODELS:
        synthetic, stats = generate_with_synthcity(
            train,
            test=test,
            metadata=metadata,
            model_name=model_name,
            n_samples=n_samples,
            random_seed=GENERATION_SEED,
            target_column=metadata.target_column,
            minority_class=minority_class,
            fold_id=fold_id,
            output_root=OUTPUT_ROOT,
        )

        stats.insert(0, "fold", fold_id)
        all_stats.append(stats)

        # Keep completed runs if a later model fails.
        pd.concat(all_stats, ignore_index=True).to_json(
            report_path, orient="records", indent=2
        )

        print(
            f"Fold {fold_id} | "
            f"Model: {model_name} | "
            f"Minority: {minority_class} | "
            f"Generated: {len(synthetic)}"
        )

        display(stats)

[2026-10-15T12:04:16.140853-0300][714443][CRITICAL] module disabled: /home/labian-3/miniconda3/envs/tdallm-synthcity/lib/python3.10/site-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch: 100%|██████████| 1000/1000 [08:48<00:00,  1.89it/s, loss=0.959]


ddpm | fold 0 | attempt 1/10 | generated 20038 | wrong class 0 | duplicates 9 | accumulated 1.10  classes {'>50K': 20038} | copies of train 0
Fold 0 | Model: ddpm | Minority: >50K | Generated: 18216


,fold,model,minority_class,random_seed,generation_seed,needed,requested,generated,valid,wrong_class,duplicates,excess,attempts,copies_train,copies_test
0,0,ddpm,>50K,42,42,18216,20038,20038,20029,0,9,1813,1,0,0


[2026-10-15T12:14:11.259048-0300][714443][CRITICAL] module disabled: /home/labian-3/miniconda3/envs/tdallm-synthcity/lib/python3.10/site-packages/synthcity/plugins/generic/plugin_goggle.py
 35%|███▍      | 699/2000 [33:32<1:02:25,  2.88s/it]


ctgan | fold 0 | attempt 1/10 | generated 20038 | wrong class 278 | duplicates 0 | accumulated 1.08  classes {'>50K': 19760, '<=50K': 278} | copies of train 0
Fold 0 | Model: ctgan | Minority: >50K | Generated: 18216


...

ctgan | fold 4 | attempt 1/10 | generated 20040 | wrong class 323 | duplicates 0 | accumulated 1.08  classes {'>50K': 19717, '<=50K': 323} | copies of train 0
Fold 4 | Model: ctgan | Minority: >50K | Generated: 18218


,fold,model,minority_class,random_seed,generation_seed,needed,requested,generated,valid,wrong_class,duplicates,excess,attempts,copies_train,copies_test
0,4,ctgan,>50K,42,42,18218,20040,20040,19717,323,0,1499,1,0,0


[2026-10-15T17:11:35.827745-0300][714443][CRITICAL] module disabled: /home/labian-3/miniconda3/envs/tdallm-synthcity/lib/python3.10/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|██████████| 1000/1000 [31:05<00:00,  1.87s/it]


tvae | fold 4 | attempt 1/10 | generated 20040 | wrong class 0 | duplicates 0 | accumulated 1.10  classes {'>50K': 20040} | copies of train 0
Fold 4 | Model: tvae | Minority: >50K | Generated: 18218


,fold,model,minority_class,random_seed,generation_seed,needed,requested,generated,valid,wrong_class,duplicates,excess,attempts,copies_train,copies_test
0,4,tvae,>50K,42,42,18218,20040,20040,20040,0,0,1822,1,0,0


In [32]:
for fold_id, (train, test, audit) in enumerate(folds):
    for method in CLASSICAL_METHODS:
        actual_method = method
        if method == "smotenc" and not any(
            column in train.columns
            for column in metadata.categorical_columns
        ):
            actual_method = "smote"

        balanced_train, stats = balance_data(
            train,
            test=test,
            metadata=metadata,
            method=method,
            fold_id=fold_id,
            random_seed=GENERATION_SEED,
            output_root=PROJECT_ROOT / "artifacts" / "generation"
        )

        stats.insert(0, "fold", fold_id)
        all_stats.append(stats)

        # Keep one report with generative and classical baseline statistics.
        pd.concat(all_stats, ignore_index=True).to_json(
            report_path, orient="records", indent=2
        )

        print(
            f"Fold {fold_id} | "
            f"Method: {actual_method} | "
            f"Rows saved: {len(balanced_train)}"
        )

Fold 0 | Method: ros | Rows saved: 18216
Fold 0 | Method: rus | Rows saved: 8962
Fold 0 | Method: smotenc | Rows saved: 18216
Fold 1 | Method: ros | Rows saved: 18216
Fold 1 | Method: rus | Rows saved: 8962
Fold 1 | Method: smotenc | Rows saved: 18216
Fold 2 | Method: ros | Rows saved: 18216
Fold 2 | Method: rus | Rows saved: 8962
Fold 2 | Method: smotenc | Rows saved: 18216
Fold 3 | Method: ros | Rows saved: 18218
Fold 3 | Method: rus | Rows saved: 8961
Fold 3 | Method: smotenc | Rows saved: 18218
Fold 4 | Method: ros | Rows saved: 18218
Fold 4 | Method: rus | Rows saved: 8961
Fold 4 | Method: smotenc | Rows saved: 18218
